# Fundamentos y simulación básica

Primer recorrido por ProteinMotorSim: definición del motor, condiciones experimentales, red cinética, simulación estocástica y análisis de una trayectoria.

> **Nota científica:** ProteinMotorSim es un prototipo de modelado cinético fenomenológico. Los parámetros iniciales son ilustrativos y los mapas de energía no están calibrados experimentalmente para kinesina.

In [ ]:
from pathlib import Path
import sys

# Busca la raíz del proyecto para poder importar src/protein_motor_sim.
PROJECT_ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "protein_motor_sim").is_dir():
        PROJECT_ROOT = candidate
        sys.path.insert(0, str(candidate / "src"))
        break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "No encuentro la carpeta src/protein_motor_sim. "
        "Abre JupyterLab desde la raíz de ProteinMotorSim o instala el paquete "
        "con python -m pip install -e ."
    )

print(f"Proyecto detectado en: {PROJECT_ROOT}")

## 1. Objetivo

En este notebook construiremos una simulación individual, inspeccionaremos la red cinética y analizaremos la trayectoria. Las unidades empleadas son: tiempo (s), posición (nm), fuerza (N) y concentración de ATP (mol/L).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from protein_motor_sim import Environment, GillespieSimulator, Kinesin1
from protein_motor_sim.analysis.observables import analyze_trajectory
from protein_motor_sim.visualization.plots import plot_trajectory
from protein_motor_sim.visualization.molecular_view import plot_kinesin_schematic

## 2. Definir el motor y el entorno

`Kinesin1` proporciona los parámetros y la red de estados. `Environment` define las condiciones experimentales. La fuerza externa positiva se interpreta como una carga que se opone al avance hacia el extremo positivo.

In [ ]:
motor = Kinesin1()
environment = Environment(
    atp_concentration=1.0e-3,  # 1 mM
    external_force_N=0.0,      # N
    temperature_K=310.0,       # K
)
simulator = GillespieSimulator()

print("Motor:", motor.name)
print("Paso nominal (nm):", motor.step_size_nm)
print("Entorno:", environment)
print("Parámetros:", motor.parameters)

## 3. Inspeccionar la red de reacciones

La red actual agrupa el mecanismo en tres estados activos (`bound_apo`, `atp_bound`, `post_hydrolysis`) y un estado terminal (`detached`). No representa explícitamente las dos cabezas ni todos los intermediarios estructurales de kinesina.

In [ ]:
network = motor.build_network()
print("Estado inicial:", network.initial_state)
print("\nEstados:")
for name, state in network.states.items():
    print(f"- {name}: {state.description}; terminal={state.is_terminal}")

print("\nTransiciones y tasas disponibles desde cada estado:")
for state_name in network.states:
    print(f"\n{state_name}")
    for transition, rate in network.rates(state_name, environment):
        print(f"  {transition.event_name:20s} -> {transition.target:18s} {rate:10.4f} s^-1")

## 4. Ejecutar una trayectoria de Gillespie

La semilla permite reproducir exactamente la misma realización aleatoria. Cambia `t_max` o `seed` y vuelve a ejecutar para explorar otras trayectorias.

In [ ]:
trajectory = simulator.run(
    motor=motor,
    environment=environment,
    t_max=0.5,
    seed=2026,
)

print("Número de registros:", len(trajectory.times_s))
print("Eventos:", trajectory.number_of_events)
print("Estado final:", trajectory.final_state)
print("ATP consumido:", trajectory.atp_consumed_total)
print("Desplazamiento neto (nm):", trajectory.displacement_nm)

## 5. Analizar y visualizar

La velocidad calculada es la velocidad media neta de la trayectoria registrada. El `run_length_nm` del analizador actual es el valor absoluto del desplazamiento neto; no debe confundirse automáticamente con todas las definiciones experimentales de longitud de recorrido o processividad.

In [ ]:
summary = analyze_trajectory(trajectory)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
plot_trajectory(trajectory, ax=ax)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
plot_kinesin_schematic(position_nm=trajectory.positions_nm[-1], ax=ax)
plt.show()

In [ ]:
trajectory_table = pd.DataFrame({
    "time_s": trajectory.times_s,
    "position_nm": trajectory.positions_nm,
    "state": trajectory.states,
    "event": trajectory.events,
})
display(trajectory_table.head(12))

## 6. Interpretación

- Gillespie no calcula una trayectoria continua: genera una secuencia de eventos y tiempos aleatorios.
- La posición cambia cuando ocurre un evento con desplazamiento distinto de cero.
- El ATP se contabiliza en el evento `atp_hydrolysis`. En esta red simplificada no existe todavía un acoplamiento termodinámico completo entre cada ATP y cada paso mecánico.
- La figura molecular es esquemática, no una estructura atomística.